In [2]:
#Downloading dataset:
import kagglehub
import pandas as pd
from pathlib import Path

path = kagglehub.dataset_download("subigyanepal/college-experience-dataset") 
print("Path to dataset files:", path)

ReadTimeout: HTTPSConnectionPool(host='www.kaggle.com', port=443): Read timed out. (read timeout=5)

In [3]:
#Basepath used to load the dataset files:
base_path = Path(path)
print("Base path:", base_path)

Base path: /Users/sethunair/.cache/kagglehub/datasets/subigyanepal/college-experience-dataset/versions/5


In [4]:
#Function to load data from a specified file and return a dataframe:
def load_data(file_path):
    """Loads data from the specified file path and returns a DataFrame."""
    try:
    #Check path and load data:
        print(f"Loading data from: {file_path}")
        data = pd.read_csv(file_path)
        print(f"Successfully loaded data from {file_path}")
    
    #Rename dataframe according to the file name:
        file_name = file_path.stem  # Get the file name without extension
        data.name = file_name  # Assign the file name as an attribute to the dataframe  
    
    #Name dataframe columns and print the first few rows of the data:
        print(data.head())
        print("Data columns:", data.columns)

        return data

    except Exception as e:
        print(f"Error loading data from {file_path}: {e}")
        return None

In [5]:
#Function to relabel each unique ID with a new integer label:
def relabel_ids(df, id_column):
    """Maps each unque ID in the specified column to a new integer label."""
    unique_ids = df[id_column].unique()
    id_mapping = {old_id: new_id for new_id, old_id in enumerate(unique_ids, start=1)}
    df[id_column] = df[id_column].map(id_mapping)
    return df

In [6]:
#Loading the EMA data:
EMA_path = base_path / "EMA" / "general_ema.csv"
EMA_data = load_data(EMA_path)

Loading data from: /Users/sethunair/.cache/kagglehub/datasets/subigyanepal/college-experience-dataset/versions/5/EMA/general_ema.csv
Successfully loaded data from /Users/sethunair/.cache/kagglehub/datasets/subigyanepal/college-experience-dataset/versions/5/EMA/general_ema.csv
                                uid       day  pam  phq4-1  phq4-2  phq4-3  \
0  1ff6d7f34acb354430e7323a35ff7703  20170907  7.0     2.0     2.0     2.0   
1  1ff6d7f34acb354430e7323a35ff7703  20170908  3.0     1.0     1.0     1.0   
2  1ff6d7f34acb354430e7323a35ff7703  20170909  NaN     NaN     NaN     NaN   
3  1ff6d7f34acb354430e7323a35ff7703  20170910  NaN     NaN     NaN     NaN   
4  1ff6d7f34acb354430e7323a35ff7703  20170911  2.0     1.0     1.0     1.0   

   phq4-4  phq4_resp_mean  phq4_resp_median  phq4_score  social_level  sse3-1  \
0     1.0        4.494280          4.409224         7.0           4.0     1.0   
1     1.0        4.634802          3.842536         4.0           2.0     2.0   
2     NaN  

In [7]:
#Loading the sensing data:
sensing_path = base_path / "Sensing" / "sensing.csv"
sensing_data = load_data(sensing_path)

Loading data from: /Users/sethunair/.cache/kagglehub/datasets/subigyanepal/college-experience-dataset/versions/5/Sensing/sensing.csv
Successfully loaded data from /Users/sethunair/.cache/kagglehub/datasets/subigyanepal/college-experience-dataset/versions/5/Sensing/sensing.csv
                                uid  is_ios       day  act_in_vehicle_ep_0  \
0  1ff6d7f34acb354430e7323a35ff7703       1  20170907                    0   
1  1ff6d7f34acb354430e7323a35ff7703       1  20170908                    0   
2  1ff6d7f34acb354430e7323a35ff7703       1  20170909                  110   
3  1ff6d7f34acb354430e7323a35ff7703       1  20170910                    0   
4  1ff6d7f34acb354430e7323a35ff7703       1  20170911                    0   

   act_in_vehicle_ep_1  act_in_vehicle_ep_2  act_in_vehicle_ep_3  \
0                    0                    0                    0   
1                    0                    0                    0   
2                    0                  110       

In [8]:
#Remove columns in sensing data that are not needed for analysis:
#Remove columns containing 'hr' in their name:
hr_columns = [col for col in sensing_data.columns if '_hr_' in col]
sensing_data = sensing_data.drop(columns=hr_columns)
print(sensing_data.columns)
print(sensing_data.head())

Index(['uid', 'is_ios', 'day', 'act_in_vehicle_ep_0', 'act_in_vehicle_ep_1',
       'act_in_vehicle_ep_2', 'act_in_vehicle_ep_3', 'act_on_bike_ep_0',
       'act_on_bike_ep_1', 'act_on_bike_ep_2',
       ...
       'sms_out_num_ep_3', 'unlock_duration_ep_0', 'unlock_duration_ep_1',
       'unlock_duration_ep_2', 'unlock_duration_ep_3', 'unlock_num_ep_0',
       'unlock_num_ep_1', 'unlock_num_ep_2', 'unlock_num_ep_3',
       'sleep_heathkit_dur'],
      dtype='object', length=171)
                                uid  is_ios       day  act_in_vehicle_ep_0  \
0  1ff6d7f34acb354430e7323a35ff7703       1  20170907                    0   
1  1ff6d7f34acb354430e7323a35ff7703       1  20170908                    0   
2  1ff6d7f34acb354430e7323a35ff7703       1  20170909                  110   
3  1ff6d7f34acb354430e7323a35ff7703       1  20170910                    0   
4  1ff6d7f34acb354430e7323a35ff7703       1  20170911                    0   

   act_in_vehicle_ep_1  act_in_vehicle_ep_2  a

In [9]:
#iOS vs Android distribution in the sensing data (0=Android, 1=iOS):
ios_android_counts = sensing_data['is_ios'].value_counts()
print("iOS vs Android distribution:")
print(ios_android_counts)

iOS vs Android distribution:
is_ios
1    190960
0     25105
Name: count, dtype: int64


In [10]:
#Loading the demographics: 
demographics_path = base_path / "Demographics" / "demographics.csv"
demographics_data = load_data(demographics_path)

Loading data from: /Users/sethunair/.cache/kagglehub/datasets/subigyanepal/college-experience-dataset/versions/5/Demographics/demographics.csv
Successfully loaded data from /Users/sethunair/.cache/kagglehub/datasets/subigyanepal/college-experience-dataset/versions/5/Demographics/demographics.csv
                                uid gender   race
0  3569e2f520db9014b4acc4227a6421c1   both  white
1  ac70fe1f8115ac361f2023269c011c3e      M  asian
2  3bb377ba0acb7d8916010184df36aa57      F  white
3  fa394f6d3d077bd5568fc3bc01580806      F  white
4  84120765740b5395aa49a2feb12fbb43      M  asian
Data columns: Index(['uid', 'gender', 'race'], dtype='object')


In [11]:
#Demogrpahics data, race and gender:
print("Race distribution:")
print(demographics_data['race'].value_counts())
print("\nGender distribution:")
print(demographics_data['gender'].value_counts())  

Race distribution:
race
white                            135
asian                             53
more than one                     10
black                              8
other/hispanic                     6
american indian/alaska native      2
american indian/white              1
alaskan native/white               1
Name: count, dtype: int64

Gender distribution:
gender
F       146
M        69
both      1
Name: count, dtype: int64


In [12]:
#Merge the datasets on the common 'id' and 'timestamp' columns:
merged_data = pd.merge(EMA_data, sensing_data, on=['uid', 'day'], how='inner')
merged_data = pd.merge(merged_data, demographics_data, on='uid', how='inner')
print("Merged data shape:", merged_data.shape)
print("Merged data columns:", merged_data.columns)
print(merged_data.head())

Merged data shape: (214751, 190)
Merged data columns: Index(['uid', 'day', 'pam', 'phq4-1', 'phq4-2', 'phq4-3', 'phq4-4',
       'phq4_resp_mean', 'phq4_resp_median', 'phq4_score',
       ...
       'unlock_duration_ep_1', 'unlock_duration_ep_2', 'unlock_duration_ep_3',
       'unlock_num_ep_0', 'unlock_num_ep_1', 'unlock_num_ep_2',
       'unlock_num_ep_3', 'sleep_heathkit_dur', 'gender', 'race'],
      dtype='object', length=190)
                                uid       day  pam  phq4-1  phq4-2  phq4-3  \
0  1ff6d7f34acb354430e7323a35ff7703  20170907  7.0     2.0     2.0     2.0   
1  1ff6d7f34acb354430e7323a35ff7703  20170908  3.0     1.0     1.0     1.0   
2  1ff6d7f34acb354430e7323a35ff7703  20170909  NaN     NaN     NaN     NaN   
3  1ff6d7f34acb354430e7323a35ff7703  20170910  NaN     NaN     NaN     NaN   
4  1ff6d7f34acb354430e7323a35ff7703  20170911  2.0     1.0     1.0     1.0   

   phq4-4  phq4_resp_mean  phq4_resp_median  phq4_score  ...  \
0     1.0        4.494280      

In [13]:
#Relabel the 'id' column in the merged dataset with new integer labels:
merged_data = relabel_ids(merged_data, 'uid')
print("Merged data with relabeled IDs:")
print(merged_data.head())

#Count of unique IDs in the merged dataset:
unique_id_count = merged_data['uid'].nunique()
print(f"Number of unique IDs in the merged dataset: {unique_id_count}")

Merged data with relabeled IDs:
   uid       day  pam  phq4-1  phq4-2  phq4-3  phq4-4  phq4_resp_mean  \
0    1  20170907  7.0     2.0     2.0     2.0     1.0        4.494280   
1    1  20170908  3.0     1.0     1.0     1.0     1.0        4.634802   
2    1  20170909  NaN     NaN     NaN     NaN     NaN             NaN   
3    1  20170910  NaN     NaN     NaN     NaN     NaN             NaN   
4    1  20170911  2.0     1.0     1.0     1.0     1.0      333.580775   

   phq4_resp_median  phq4_score  ...  unlock_duration_ep_1  \
0          4.409224         7.0  ...                 0.000   
1          3.842536         4.0  ...               105.725   
2               NaN         NaN  ...               682.900   
3               NaN         NaN  ...               376.235   
4          2.885290         4.0  ...               548.430   

   unlock_duration_ep_2  unlock_duration_ep_3  unlock_num_ep_0  \
0              2091.008              2777.992               62   
1              3261.310 

In [14]:
#Missing values in stress column:
missing_stress_count = merged_data['stress'].isnull().sum()
print(f"Number of missing values in 'stress' column: {missing_stress_count}")

Number of missing values in 'stress' column: 179625


In [15]:
#Drop rows with missing stress values:
merged_data = merged_data.dropna(subset=['stress'])
print(f"\nData shape after dropping rows with missing stress values: {merged_data.shape}")

#See how many stress levels are in the merged dataset:
print("\nStress level distribution:")
print(merged_data['stress'].value_counts())   


Data shape after dropping rows with missing stress values: (35126, 190)

Stress level distribution:
stress
2.0    11879
3.0    10027
1.0     6647
4.0     4811
5.0     1762
Name: count, dtype: int64


In [16]:
#One-hot encode gender and map to binary values (0 for F, 1 for M):
merged_data["gender"] = merged_data["gender"].map({"F": 0, "M": 1})
print("\nGender distribution after mapping to binary values:")
print(merged_data['gender'].value_counts())


Gender distribution after mapping to binary values:
gender
0.0    23720
1.0    11131
Name: count, dtype: int64


In [17]:
#One-hot enconding race and map to binary values:
#threshold for including race data in 'other':
threshold = 5
counts = merged_data["race"].value_counts()

merged_data["race"] = merged_data["race"].where(merged_data["race"].map(counts) >= threshold, "Other")

race_dummies = pd.get_dummies(merged_data["race"], prefix="race")
race_dummies = race_dummies.astype(int) # Convert to integer type (0/1)
merged_data = pd.concat([merged_data.drop(columns=["race"]), race_dummies], axis=1)

merged_data.head()


,uid,day,pam,phq4-1,phq4-2,phq4-3,phq4-4,phq4_resp_mean,phq4_resp_median,phq4_score,...,sleep_heathkit_dur,gender,race_alaskan native/white,race_american indian/alaska native,race_american indian/white,race_asian,race_black,race_more than one,race_other/hispanic,race_white
0,1,20170907,7.0,2.0,2.0,2.0,1.0,4.494280,4.409224,7.0,...,NaN,0.0,0,0,0,1,0,0,0,0
1,1,20170908,3.0,1.0,1.0,1.0,1.0,4.634802,3.842536,4.0,...,8.267,0.0,0,0,0,1,0,0,0,0
4,1,20170911,2.0,1.0,1.0,1.0,1.0,333.580775,2.885290,4.0,...,8.683,0.0,0,0,0,1,0,0,0,0
10,1,20170917,10.0,1.0,1.0,1.0,1.0,10.424530,8.322366,4.0,...,8.150,0.0,0,0,0,1,0,0,0,0
19,1,20170926,1.0,1.0,1.0,1.0,1.0,2.507608,2.838959,4.0,...,8.967,0.0,0,0,0,1,0,0,0,0


In [18]:
#Preparing the final dataset for analysis:
#Select relevant columns for analysis:
y = merged_data['stress']
X = merged_data.drop(columns=['stress', 'uid', 'day'])  # Drop target, ID columns, and date
print("Final feature set columns:", X.columns)
print("Final feature set shape:", X.shape)  

Final feature set columns: Index(['pam', 'phq4-1', 'phq4-2', 'phq4-3', 'phq4-4', 'phq4_resp_mean',
       'phq4_resp_median', 'phq4_score', 'social_level', 'sse3-1',
       ...
       'sleep_heathkit_dur', 'gender', 'race_alaskan native/white',
       'race_american indian/alaska native', 'race_american indian/white',
       'race_asian', 'race_black', 'race_more than one', 'race_other/hispanic',
       'race_white'],
      dtype='object', length=194)
Final feature set shape: (35126, 194)


In [26]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegressionCV

X_train, _, y_train, _ = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# subset for speed
n = min(8000, len(X_train))
X_sub = X_train.sample(n=n, random_state=42)
y_sub = y_train.loc[X_sub.index]

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegressionCV(
        solver="saga",
        penalty="elasticnet",
        l1_ratios=[1.0],           # pure L1
        cv=3,
        Cs=np.logspace(-2, 2, 6),  # fewer Cs
        max_iter=1500,
        tol=1e-3,
        scoring="accuracy",        # faster than f1_macro
        n_jobs=-1,
        # use_legacy_attributes=False
    ))
])

pipe.fit(X_sub, y_sub)

coef = pd.DataFrame(pipe.named_steps["clf"].coef_, columns=X.columns)
selected = (coef.abs() > 1e-12).any(axis=0)
selected_features = coef.columns[selected].tolist()

print("Selected:", len(selected_features), "/", X.shape[1])

/Users/sethunair/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/sethunair/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/sethunair/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/sethunair/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/sethunair/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/sethunair/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/sethunair/Library/Python/3.9/lib/python/site-packages/sklea

Selected: 175 / 194


In [27]:
# coef shape = (n_classes, n_features)
coef = pd.DataFrame(pipe.named_steps["clf"].coef_, columns=X.columns)

# importance: biggest effect in any class
importance = coef.abs().max(axis=0).sort_values(ascending=False)

print("Top 30 features by max |coef| across classes:")
print(importance.head(30))

# (optional) show which class drives each top feature
top = importance.head(30).index
which_class = coef.loc[:, top].abs().idxmax(axis=0)      # row index = class number
sign = coef.loc[:, top].apply(lambda col: col.iloc[col.abs().argmax()])  # signed coef

summary = pd.DataFrame({
    "max_abs_coef": importance.loc[top],
    "class_with_max": which_class.values,
    "signed_coef_at_max": sign.values
}).sort_values("max_abs_coef", ascending=False)

print("\nTop 30 with class + signed coefficient:")
print(summary)

Top 30 features by max |coef| across classes:
sse3-1                       0.960281
phq4-1                       0.507774
phq4_score                   0.415810
sse3-3                       0.403018
sse3-4                       0.351498
phq4-2                       0.272287
loc_self_dorm_dur            0.183537
sleep_end                    0.181921
race_other/hispanic          0.166693
social_level                 0.166609
race_white                   0.153217
loc_study_dur                0.151771
loc_food_audio_amp           0.143921
pam                          0.123925
sse3_resp_median             0.121363
quality_loc                  0.109281
act_walking_ep_0             0.102386
unlock_num_ep_2              0.101312
phq4-4                       0.099590
loc_social_dur               0.098896
quality_audio                0.096120
sleep_duration               0.092449
act_walking_ep_2             0.090441
act_in_vehicle_ep_0          0.088559
light_std_ep_2               0.077145
sse3

In [ ]:
#Top Correlations
num = merged_data.select_dtypes(include="number").copy()
corr = num.corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
pairs = corr.where(mask).stack().rename("corr")
print("\nTop 5 strongest correlations (signed):")
print(pairs.reindex(pairs.abs().sort_values(ascending=False).index).head(15).round(3))


Top 5 strongest correlations (signed):
loc_home_audio_amp            loc_self_dorm_audio_amp          1.000
loc_home_convo_duration       loc_self_dorm_convo_duration     1.000
loc_home_unlock_duration      loc_self_dorm_unlock_duration    0.999
loc_home_audio_voice          loc_self_dorm_audio_voice        0.999
loc_home_unlock_num           loc_self_dorm_unlock_num         0.999
loc_max_dis_from_campus_ep_0  loc_max_dis_from_campus_ep_2     0.996
loc_home_still                loc_self_dorm_still              0.996
loc_max_dis_from_campus_ep_0  loc_max_dis_from_campus_ep_1     0.995
loc_max_dis_from_campus_ep_1  loc_max_dis_from_campus_ep_2     0.995
loc_max_dis_from_campus_ep_2  loc_max_dis_from_campus_ep_3     0.993
loc_home_convo_num            loc_self_dorm_convo_num          0.993
loc_max_dis_from_campus_ep_0  loc_max_dis_from_campus_ep_3     0.992
loc_max_dis_from_campus_ep_1  loc_max_dis_from_campus_ep_3     0.987
sse3_resp_mean                avg_ema_spent_time               

In [35]:
#find number of columns with home in the name:
home_columns = [col for col in merged_data.columns if 'home' in col]
print(f"\nNumber of columns with 'home' in the name: {len(home_columns)}")
print(home_columns)
#drop columns with home in the name:
merged_data = merged_data.drop(columns=home_columns)
# #top correlations again after dropping home columns:
# num = merged_data.select_dtypes(include="number").copy()
# corr = num.corr()
# mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
# pairs = corr.where(mask).stack().rename("corr")
# print("\nTop 5 strongest correlations (signed) after dropping 'home' columns:")
# print(pairs.reindex(pairs.abs().sort_values(ascending=False).index).head(15).round(3))

#drop columns max_dis_from_campus_ep_1, max_dis_from_campus_ep_2, max_dis_from_campus_ep_3:
merged_data = merged_data.drop(columns=['loc_max_dis_from_campus_ep_1', 'loc_max_dis_from_campus_ep_2', 'loc_max_dis_from_campus_ep_3'])
#top correlations again after dropping max_dis_from_campus_ep columns and home columns:
num = merged_data.select_dtypes(include="number").copy()
corr = num.corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
pairs = corr.where(mask).stack().rename("corr")
print("\nTop 15 strongest correlations (signed) after dropping 'home' and 'max_dis_from_campus_ep' columns:")
print(pairs.reindex(pairs.abs().sort_values(ascending=False).index).head(15).round(3))



Number of columns with 'home' in the name: 0
[]

Top 15 strongest correlations (signed) after dropping 'home' and 'max_dis_from_campus_ep' columns:
sse3_resp_mean             avg_ema_spent_time           0.959
audio_amp_mean_ep_0        audio_amp_std_ep_0           0.945
light_std_ep_0             light_std_ep_2               0.938
audio_amp_mean_ep_3        audio_amp_std_ep_3           0.936
audio_convo_num_ep_0       audio_convo_num_ep_2         0.933
audio_amp_mean_ep_2        audio_amp_std_ep_2           0.928
light_mean_ep_0            light_mean_ep_2              0.925
audio_convo_duration_ep_0  audio_convo_num_ep_0         0.919
sms_in_num_ep_2            sms_out_num_ep_2             0.918
sms_in_num_ep_0            sms_out_num_ep_0             0.914
audio_convo_duration_ep_0  audio_convo_duration_ep_2    0.909
unlock_num_ep_0            unlock_num_ep_2              0.908
act_in_vehicle_ep_0        act_in_vehicle_ep_2          0.908
sms_in_num_ep_3            sms_out_num_ep_3  